In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
from datetime import timedelta
import math

pd.set_option('display.max_columns',None)

In [10]:
os.getcwd()

'D:\\Machine Learning\\projects\\E_commerce\\notebooks\\data working steps'

In [2]:
def get_dataset(dataset:str)->str:
    try:
        return pd.read_csv(os.path.join('..','..','data','processed',f'{dataset}.csv'))
    except Exception as e:
        warnings.warn(f'"{dataset}" dataset not present.')
        return None

In [3]:
full_data = get_dataset('merged_data')

In [4]:
training_df = get_dataset('training_data')

In [5]:
training_df.isna().sum()

['order_id','order_purchase_timestamp','order_delivered_customer_date',
'payment_type','customer_state','customer_latitude',
'customer_longitude','price','freight_value','product_weight_g',
'seller_state','seller_latitude','seller_longitude']

['order_id',
 'order_purchase_timestamp',
 'order_delivered_customer_date',
 'payment_type',
 'customer_state',
 'customer_latitude',
 'customer_longitude',
 'price',
 'freight_value',
 'product_weight_g',
 'seller_state',
 'seller_latitude',
 'seller_longitude']

In [47]:


import pandas as pd
import numpy as np
import os
# from sklearn.model_selection import
from datetime import timedelta

def calculate_distance(record):
    lat1 = np.radians( record['customer_latitude'])
    lng1 = np.radians(record['customer_longitude'])
    lat2 = np.radians(record['seller_latitude'])
    lng2 = np.radians(record['seller_longitude'])

    # Calucate the difference.
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    
    a = (np.sin(dlat/2)**2) + np.cos(lat1) * np.cos(lat2) * np.sin(dlng/2)**2

    c =  2 * np.asin(np.sqrt(a))

    r = 6371

    return np.round((c * r),0)
    

class Training_pipeline():

   

    def __init__(self,data):
        self.data = data

    
    def __clean_data(self):
        # Convert columns to their appropriate datatypes.

        date_col = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
            'order_delivered_customer_date','order_estimated_delivery_date','shipping_limit_date']

        number_col = ['payment_sequential','payment_installments','order_item_id','seller_zip_code_prefix']

        for col in date_col:
            self.data[col] = pd.to_datetime( self.data[col])
            

        for col in number_col:
            self.data[col] = self.data[col].astype('Int64')

        imp_features = ['order_id','order_purchase_timestamp','order_delivered_customer_date',
                'payment_type','customer_state','customer_latitude',
                'customer_longitude','price','freight_value','product_weight_g',
                'seller_state','seller_latitude','seller_longitude']
        self.data = self.data[imp_features]      

        # Those zip code are not present in geolocation table so we need to drop them.
        self.data.dropna(axis=0,subset=['customer_latitude','customer_longitude','seller_latitude','seller_longitude'],inplace=True)
        self.data = self.data.dropna(subset=['order_delivered_customer_date'])
        self.data.dropna(inplace=True)

        # adding output feature
        self.__add_output_feature()
        upper_lim = self.data['delivery_days'].quantile(0.99)
        self.data = self.data[  self.data['delivery_days']<upper_lim]


        # adding new feature.
        self.__add_distance()

        self.__feature_engg()

        # filtering records.
        self.data = self.data[(self.data['delivery_distance_km']>25) & (self.data['delivery_days']>1)]        

      

    def __add_distance(self):
        
       self.data['delivery_distance_km']  = self.data.apply(calculate_distance, axis=1)

    
    def __add_output_feature(self):
        self.data['delivery_days'] = (self.data['order_delivered_customer_date'] - self.data['order_purchase_timestamp']).dt.days


    def __feature_engg(self):
        
        self.data['is_same_state']= np.where((self.data['customer_state'].str.lower()) == (self.data['seller_state'].str.lower()),1,0 )

    def __transform(self):
        cols = ['delivery_days' ,'delivery_distance_km' ,'freight_value' ]
        for col in cols:
            self.data[f'{col}_log']= np.log1p(self.data[col])


    def get_data(self):
        '''
        get the processed features and targets,
        return tuple: features, target.
        '''
        data_groupby = self.data.groupby('order_id').agg(  
                                delivery_days = ('delivery_days_log','first'), 
                                order_count = ('seller_state','nunique'), 
                              # min_distance = ('delivery_distance_km_log','min'),
                              # max_distance=('delivery_distance_km_log','max') ,
                                avg_distance = ('delivery_distance_km_log','mean'),
                                 # purchase_year = ('purchase_year_log', 'first'),
                                 is_same_state = ('is_same_state','first'),
                                 
                                 freight_value = ('freight_value_log','sum')                             
                              
                            
                            )
        output_data = data_groupby.reset_index().drop(columns = 'order_id')
        return (output_data.drop(columns='delivery_days'),output_data['delivery_days'])
        
    def process(self):
        self.__clean_data()  
        self.__add_distance()
        self.__add_output_feature()
        self.__feature_engg()
        self.__transform()
     
        
   


In [49]:
tp =  Training_pipeline(get_dataset('training_data'))

In [50]:
tp.process()

In [58]:
data.head()

AttributeError: 'NoneType' object has no attribute 'head'

In [59]:
data['purchase_year'] = (data['order_purchase_timestamp'].dt.year)-2016


data['is_same_state']= np.where((data['customer_state'].str.lower()) == (data['seller_state'].str.lower()),1,0 )

TypeError: 'NoneType' object is not subscriptable

In [60]:
data = data[(data['delivery_distance_km']>25) & (data['delivery_days']>1)]


TypeError: 'NoneType' object is not subscriptable

In [61]:
upper_lim = data['delivery_days'].quantile(0.99)
data = data[  data['delivery_days']<upper_lim]

TypeError: 'NoneType' object is not subscriptable

In [62]:
cols = ['delivery_days' ,'purchase_year' ,'delivery_distance_km' ,'freight_value' ,'product_weight_g']
for col in cols:
    data[f'{col}_log']= np.log1p(data[col])


TypeError: 'NoneType' object is not subscriptable

In [63]:
final_data = data.groupby('order_id').agg(    delivery_days = ('delivery_days_log','first'), 
                                order_count = ('seller_state','nunique'), 
                              # min_distance = ('delivery_distance_km_log','min'),
                              # max_distance=('delivery_distance_km_log','max') ,
                                avg_distance = ('delivery_distance_km_log','mean'),
                                 purchase_year = ('purchase_year_log', 'first'),
                                 is_same_state = ('is_same_state','first'),
                                 
                                 freight_value = ('freight_value_log','sum')
                                 
                              
                            
                            ).sort_values(by='order_count',ascending=False)

AttributeError: 'NoneType' object has no attribute 'groupby'

In [64]:
from sklearn.model_selection import train_test_split

In [65]:
train,test = train_test_split(final_data,test_size=0.25)

NameError: name 'final_data' is not defined

In [66]:
train.shape,test.shape

NameError: name 'train' is not defined

In [67]:
x_train = train.drop(columns=['delivery_days'])
y_train = train['delivery_days']

x_test = test.drop(columns=['delivery_days'])
y_test = test['delivery_days']

NameError: name 'train' is not defined

In [68]:
x_train.shape,y_train.shape, x_test.shape, y_test.shape

NameError: name 'x_train' is not defined

In [69]:
train.columns.to_list()

NameError: name 'train' is not defined

In [ ]:
plt.figure(figsize=(14, 10))
corr_matrix = train.select_dtypes(include='number').corr()

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5
)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Baseline: always predict mean
y_pred_baseline = np.full(len(y_test), y_train.mean())

# Evaluate on actual days (inverse transform)
baseline_mae = mean_absolute_error(
    np.expm1(y_test), 
    np.expm1(y_pred_baseline)
)
print(f"Baseline MAE: {baseline_mae:.2f} days")

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(x_train, y_train)

y_pred_lr = lr.predict(x_test)

lr_mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred_lr))
lr_r2  = r2_score(np.expm1(y_test), np.expm1(y_pred_lr))

print(f"Linear Regression MAE: {lr_mae:.2f} days")
print(f"Linear Regression R²:  {lr_r2:.3f}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf.fit(x_train, y_train)

y_pred_rf = rf.predict(x_test)

rf_mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred_rf))
rf_r2  = r2_score(np.expm1(y_test), np.expm1(y_pred_rf))

print(f"Random Forest MAE: {rf_mae:.2f} days")
print(f"Random Forest R²:  {rf_r2:.3f}")

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)
xgb.fit(x_train, y_train)

y_pred_xgb = xgb.predict(x_test)

xgb_mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred_xgb))
xgb_r2  = r2_score(np.expm1(y_test), np.expm1(y_pred_xgb))

print(f"XGBoost MAE: {xgb_mae:.2f} days")
print(f"XGBoost R²:  {xgb_r2:.3f}")

In [ ]:
results = pd.DataFrame({
    'Model': ['Baseline', 'Linear Regression', 'Random Forest', 'XGBoost'],
    'MAE (days)': [baseline_mae, lr_mae, rf_mae, xgb_mae],
    'R²': [0, lr_r2, rf_r2, xgb_r2]
})

results = results.sort_values('MAE (days)')
print(results)
